In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert ".venv" in sys.executable, "Wrong kernel! Switch to 'Python (f1-tyre-strategy)' (top right)."
PROJECT_ROOT = Path.cwd().parent
assert PROJECT_ROOT.name == "f1-tyre-strategy-simulator", "Notebook isn't in the notebooks/ folder!"

RACES_DIR = PROJECT_ROOT / "data" / "raw" / "races"
QUALI_DIR = PROJECT_ROOT / "data" / "raw" / "quali"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DRY = ["SOFT", "MEDIUM", "HARD"]
WET = ["INTERMEDIATE", "WET"]
print("✅ Kernel and paths OK")

✅ Kernel and paths OK


In [2]:
laps = pd.concat([pd.read_parquet(f) for f in sorted(RACES_DIR.glob("*.parquet"))], ignore_index=True)
quali = pd.concat([pd.read_parquet(f) for f in sorted(QUALI_DIR.glob("*.parquet"))], ignore_index=True)

n_races = laps.groupby(["Year", "Round"]).ngroups
n_quali = quali.groupby(["Year", "Round"]).ngroups
print(f"Race laps: {laps.shape[0]:,} rows × {laps.shape[1]} columns | races: {n_races}")
print(f"Qualifying: {quali.shape[0]:,} driver rows | sessions: {n_quali}")

assert n_races == 92 and n_quali == 92, "Expected 92 races and 92 qualifying sessions"
print("✅ Both sources complete")

Race laps: 101,291 rows × 30 columns | races: 92
Qualifying: 1,831 driver rows | sessions: 92
✅ Both sources complete


In [3]:
# FastF1 renamed Miami's location in 2025; map it back so it's one circuit
CIRCUIT_FIXES = {"Miami Gardens": "Miami"}
laps["Circuit"] = laps["Location"].replace(CIRCUIT_FIXES)

races = laps.drop_duplicates(["Year", "Round"])[["Year", "Round", "EventName", "Circuit"]]
counts = races["Circuit"].value_counts()
print("Races per circuit:")
print(counts.to_string())

assert races["Circuit"].nunique() == 25, f"Expected 25 circuits, found {races['Circuit'].nunique()}"
assert counts["Miami"] == 4, "Miami should have 4 races (2022–2025)"
print("\n✅ 25 circuits; Miami now has 4 races")

Races per circuit:
Circuit
Sakhir               4
Jeddah               4
Yas Island           4
São Paulo            4
Mexico City          4
Austin               4
Suzuka               4
Marina Bay           4
Monza                4
Zandvoort            4
Spa-Francorchamps    4
Budapest             4
Spielberg            4
Silverstone          4
Montréal             4
Baku                 4
Monaco               4
Barcelona            4
Miami                4
Melbourne            4
Imola                3
Lusail               3
Las Vegas            3
Shanghai             2
Le Castellet         1

✅ 25 circuits; Miami now has 4 races


In [4]:
quali_cols = ["Year", "Round", "Driver", "BestQualiLapSeconds", "BestQualiCompound",
              "QualiWetTyreShare", "QualiRainfall"]

# One row per driver per race, with the results-table time from the race files
drivers = laps.drop_duplicates(["Year", "Round", "Driver"])[
    ["Year", "Round", "EventName", "Circuit", "Driver", "Team", "BestQualiSeconds"]
]
check = drivers.merge(quali[quali_cols], on=["Year", "Round", "Driver"], how="left")

both = check.dropna(subset=["BestQualiSeconds", "BestQualiLapSeconds"]).copy()
both["Diff"] = (both["BestQualiLapSeconds"] - both["BestQualiSeconds"]).round(3)

print(f"Race drivers: {len(check):,} | with both qualifying sources: {len(both):,}")
print(f"Sources identical (within 0.01 s): {(both['Diff'].abs() <= 0.01).mean():.1%}")
print(f"Differ by more than 0.1 s: {(both['Diff'].abs() > 0.1).sum()}")
print(both.loc[both["Diff"].abs() > 0.1,
               ["Year", "EventName", "Driver", "BestQualiSeconds", "BestQualiLapSeconds", "Diff"]]
      .sort_values("Diff").head(15).to_string(index=False))

print("\nOnly the results-table time:", (check["BestQualiSeconds"].notna() & check["BestQualiLapSeconds"].isna()).sum())
print("Only the lap-data time:     ", (check["BestQualiSeconds"].isna() & check["BestQualiLapSeconds"].notna()).sum())
print("Neither:                    ", (check["BestQualiSeconds"].isna() & check["BestQualiLapSeconds"].isna()).sum())

Race drivers: 1,827 | with both qualifying sources: 1,789
Sources identical (within 0.01 s): 97.1%
Differ by more than 0.1 s: 41
 Year                 EventName Driver  BestQualiSeconds  BestQualiLapSeconds    Diff
 2023  Saudi Arabian Grand Prix    SAR           128.510               89.721 -38.789
 2024      São Paulo Grand Prix    ALO            88.998               85.035  -3.963
 2023        Belgian Grand Prix    RIC           122.159              119.483  -2.676
 2025  United States Grand Prix    ALB            94.690               93.670  -1.020
 2025 Emilia Romagna Grand Prix    BEA            76.918               76.077  -0.841
 2022       Austrian Grand Prix    PER            66.143               65.404  -0.739
 2022    Mexico City Grand Prix    MSC            80.419               79.695  -0.724
 2025  United States Grand Prix    STR            94.540               93.878  -0.662
 2024     Australian Grand Prix    RIC            78.085               77.466  -0.619
 2023      

In [5]:
ref = check.copy()

# 1st choice: official results-table time
ref["QualiRef"] = ref["BestQualiSeconds"]
ref["QualiRefSource"] = np.where(ref["QualiRef"].notna(), "results", None)

# 2nd choice: best valid lap from the qualifying lap data (fixes 2025 Miami)
use_lap = ref["QualiRef"].isna() & ref["BestQualiLapSeconds"].notna()
ref.loc[use_lap, "QualiRef"] = ref.loc[use_lap, "BestQualiLapSeconds"]
ref.loc[use_lap, "QualiRefSource"] = "lap_data"

# 3rd choice: team-mate's reference (same car, same session)
teammate_ref = ref.groupby(["Year", "Round", "Team"])["QualiRef"].transform("min")
use_tm = ref["QualiRef"].isna() & teammate_ref.notna()
ref.loc[use_tm, "QualiRef"] = teammate_ref[use_tm]
ref.loc[use_tm, "QualiRefSource"] = "teammate"

print("Where each driver's qualifying reference comes from:")
print(ref["QualiRefSource"].value_counts(dropna=False).to_string())
print("\nStill missing:")
print(ref.loc[ref["QualiRef"].isna(), ["Year", "EventName", "Driver", "Team"]].to_string(index=False))

# Attach to every lap (new columns), checking no rows are gained or lost
n_before = len(laps)
laps = laps.merge(ref[["Year", "Round", "Driver", "QualiRef", "QualiRefSource"]],
                  on=["Year", "Round", "Driver"], how="left")
assert len(laps) == n_before, "Merge changed the number of laps!"
print(f"\n✅ QualiRef attached to {laps['QualiRef'].notna().mean():.1%} of laps")

Where each driver's qualifying reference comes from:
QualiRefSource
results     1789
lap_data      31
teammate       7

Still missing:
Empty DataFrame
Columns: [Year, EventName, Driver, Team]
Index: []

✅ QualiRef attached to 100.0% of laps


In [6]:
# Wet race: any race laps on intermediate or wet tyres
race_flags = (
    laps.groupby(["Year", "Round", "EventName", "Circuit"])["Compound"]
        .apply(lambda c: c.isin(WET).mean())
        .rename("RaceWetTyreShare").reset_index()
)

# Wet qualifying: any qualifying laps on intermediate or wet tyres (rain alone doesn't count)
session = quali.drop_duplicates(["Year", "Round"])[["Year", "Round", "QualiWetTyreShare", "QualiRainfall"]]
race_flags = race_flags.merge(session, on=["Year", "Round"], how="left")

race_flags["WetRace"] = race_flags["RaceWetTyreShare"] > 0
race_flags["WetQuali"] = race_flags["QualiWetTyreShare"] > 0
race_flags["ExcludeRace"] = race_flags["WetRace"] | race_flags["WetQuali"]

print(f"Wet race:   {race_flags['WetRace'].sum()}")
print(f"Wet quali:  {race_flags['WetQuali'].sum()}")
print(f"Both:       {(race_flags['WetRace'] & race_flags['WetQuali']).sum()}")
print(f"Excluded:   {race_flags['ExcludeRace'].sum()} of {len(race_flags)}")

print("\nRaces per season (total / excluded / kept):")
season = race_flags.groupby("Year").agg(Total=("ExcludeRace", "size"), Excluded=("ExcludeRace", "sum"))
season["Kept"] = season["Total"] - season["Excluded"]
print(season.to_string())

print("\nExcluded races and why:")
excl = race_flags[race_flags["ExcludeRace"]].copy()
excl["Reason"] = np.select(
    [excl["WetRace"] & excl["WetQuali"], excl["WetRace"]],
    ["wet race + wet quali", "wet race"], default="wet quali",
)
print(excl[["Year", "EventName", "Reason"]].to_string(index=False))

# --- Cross-check only (not used as a rule): median race-vs-quali gap on rough clean laps ---
rough = laps[
    laps["Compound"].isin(DRY) & laps["PitInTime"].isna() & laps["PitOutTime"].isna()
    & (laps["LapNumber"] > 1) & laps["IsAccurate"].eq(True)
    & laps["TrackStatus"].astype(str).eq("1") & laps["QualiRef"].notna()
]
median_gap = (
    (rough["LapTimeSeconds"] - rough["QualiRef"])
    .groupby([rough["Year"], rough["Round"]]).median()
    .rename("MedianGap").reset_index()
)
race_flags = race_flags.merge(median_gap, on=["Year", "Round"], how="left")

print("\nCross-check — median gap in races flagged for wet qualifying:")
print(race_flags.loc[race_flags["WetQuali"], ["Year", "EventName", "MedianGap"]]
      .sort_values("MedianGap").round(2).to_string(index=False))
print("\nCross-check — lowest median gaps among races we KEEP:")
print(race_flags.loc[~race_flags["ExcludeRace"], ["Year", "EventName", "QualiRainfall", "MedianGap"]]
      .nsmallest(8, "MedianGap").round(2).to_string(index=False))

Wet race:   12
Wet quali:  12
Both:       5
Excluded:   19 of 92

Races per season (total / excluded / kept):
      Total  Excluded  Kept
Year                       
2022     22         7    15
2023     22         4    18
2024     24         4    20
2025     24         4    20

Excluded races and why:
 Year                 EventName               Reason
 2022 Emilia Romagna Grand Prix wet race + wet quali
 2022         Monaco Grand Prix             wet race
 2022       Canadian Grand Prix            wet quali
 2022        British Grand Prix            wet quali
 2022      Singapore Grand Prix wet race + wet quali
 2022       Japanese Grand Prix             wet race
 2022      São Paulo Grand Prix            wet quali
 2023         Monaco Grand Prix             wet race
 2023       Canadian Grand Prix            wet quali
 2023        Belgian Grand Prix            wet quali
 2023          Dutch Grand Prix wet race + wet quali
 2024       Canadian Grand Prix             wet race
 2024   

In [7]:
# F1's own 107% rule: a qualifying time slower than 107% of the fastest is not representative.
pole = ref.groupby(["Year", "Round"])["QualiRef"].transform("min")
too_slow = ref["QualiRef"] > 1.07 * pole

print(f"References slower than 107% of the fastest: {too_slow.sum()}")
print(ref.loc[too_slow, ["Year", "EventName", "Driver", "QualiRef", "QualiRefSource", "BestQualiLapSeconds"]]
      .round(3).to_string(index=False))

# Fix: use the lap-data best if that one is within 107%; otherwise the team-mate's reference
lap_ok = ref["BestQualiLapSeconds"] <= 1.07 * pole
fix_lap = too_slow & lap_ok
ref.loc[fix_lap, "QualiRef"] = ref.loc[fix_lap, "BestQualiLapSeconds"]
ref.loc[fix_lap, "QualiRefSource"] = "lap_data_107"

still_slow = ref["QualiRef"] > 1.07 * pole
tm_ok = ref.loc[~still_slow].groupby(["Year", "Round", "Team"])["QualiRef"].min()
tm_vals = ref.set_index(["Year", "Round", "Team"]).index.map(tm_ok)
fix_tm = still_slow & pd.Series(tm_vals, index=ref.index).notna()
ref.loc[fix_tm, "QualiRef"] = pd.Series(tm_vals, index=ref.index)[fix_tm]
ref.loc[fix_tm, "QualiRefSource"] = "teammate_107"

print(f"\nFixed with lap data: {fix_lap.sum()} | fixed with team-mate: {fix_tm.sum()} | "
      f"still >107%: {(ref['QualiRef'] > 1.07 * pole).sum()}")
print(ref["QualiRefSource"].value_counts().to_string())

# Re-attach the corrected references to every lap
n_before = len(laps)
laps = laps.drop(columns=["QualiRef", "QualiRefSource"]).merge(
    ref[["Year", "Round", "Driver", "QualiRef", "QualiRefSource"]],
    on=["Year", "Round", "Driver"], how="left",
)
assert len(laps) == n_before, "Merge changed the number of laps!"
print("✅ Corrected references attached")

References slower than 107% of the fastest: 54
 Year                 EventName Driver  QualiRef QualiRefSource  BestQualiLapSeconds
 2022     Australian Grand Prix    STR   106.833       lap_data              106.833
 2022 Emilia Romagna Grand Prix    ALB   123.171       lap_data              123.171
 2022       Canadian Grand Prix    GAS    94.492        results               94.492
 2022       Canadian Grand Prix    PER    93.127        results               93.127
 2022       Canadian Grand Prix    LEC    93.008        results               93.008
 2022       Canadian Grand Prix    STR    95.532        results               95.532
 2022       Canadian Grand Prix    TSU    96.575        results               96.575
 2022       Canadian Grand Prix    NOR    94.066        results               94.066
 2022       Canadian Grand Prix    VET    94.512        results               94.512
 2022       Canadian Grand Prix    LAT    95.660        results               95.660
 2022      Singapo

In [8]:
# Race-level flags onto every lap
laps = laps.merge(race_flags[["Year", "Round", "WetRace", "WetQuali", "ExcludeRace"]],
                  on=["Year", "Round"], how="left")

# Track-status flag columns (TrackStatus unchanged)
status = laps["TrackStatus"].astype(str)
laps["IsSafetyCar"] = status.str.contains("4")
laps["IsVSC"] = status.str.contains("6") | status.str.contains("7")
laps["IsRedFlag"] = status.str.contains("5")
laps["IsYellow"] = status.str.contains("2")
laps["IsNeutralised"] = laps["IsSafetyCar"] | laps["IsVSC"] | laps["IsRedFlag"]

# Restart lap: the first green lap after a neutralised lap (tyres and brakes are cold)
laps = laps.sort_values(["Year", "Round", "Driver", "LapNumber"]).reset_index(drop=True)
prev_neutral = laps.groupby(["Year", "Round", "Driver"])["IsNeutralised"].shift(1, fill_value=False)
laps["IsRestartLap"] = prev_neutral & ~laps["IsNeutralised"]

# Retirement: a car about to fail can lose pace, so drop its last 2 laps
print("Most common finishing statuses:")
print(laps.drop_duplicates(["Year", "Round", "Driver"])["Status"].value_counts().head(12).to_string())
finished = laps["Status"].fillna("").str.match(r"^(Finished|Lapped|\+\d+ Laps?)$")
last_lap = laps.groupby(["Year", "Round", "Driver"])["LapNumber"].transform("max")
laps["IsRetirementLap"] = ~finished & (laps["LapNumber"] >= last_lap - 1)

rules = {
    "Wet race or wet quali":         laps["ExcludeRace"].fillna(False),
    "Lap 1 (standing start)":        laps["LapNumber"] <= 1,
    "Pit in-lap or out-lap":         laps["PitInTime"].notna() | laps["PitOutTime"].notna(),
    "SC / VSC / red-flag lap":       laps["IsNeutralised"],
    "Restart lap":                   laps["IsRestartLap"],
    "Not a dry compound":            ~laps["Compound"].isin(DRY),
    "Bad or missing timing":         ~laps["IsAccurate"].eq(True) | laps["FastF1Generated"].eq(True)
                                     | laps["LapTimeSeconds"].isna(),
    "Last 2 laps before retirement": laps["IsRetirementLap"],
}

keep = pd.Series(True, index=laps.index)
rows = []
for name, mask in rules.items():
    removed = int((keep & mask).sum())
    keep &= ~mask
    rows.append((name, removed, int(keep.sum())))

waterfall = pd.DataFrame(rows, columns=["Rule (applied in order)", "Laps removed", "Laps left"])
print(f"\nStarting laps: {len(laps):,}")
print(waterfall.to_string(index=False))
laps["PassesBasicRules"] = keep

Most common finishing statuses:
Status
Finished            1185
Lapped               297
Retired              151
+1 Lap                80
                      19
Collision damage      14
Accident              11
Collision             10
Disqualified           9
Engine                 7
+2 Laps                6
Power Unit             5

Starting laps: 101,291
      Rule (applied in order)  Laps removed  Laps left
        Wet race or wet quali         19817      81474
       Lap 1 (standing start)          1450      80024
        Pit in-lap or out-lap          4999      75025
      SC / VSC / red-flag lap          3512      71513
                  Restart lap          1222      70291
           Not a dry compound           545      69746
        Bad or missing timing            71      69675
Last 2 laps before retirement           125      69550


In [9]:
base = laps[laps["PassesBasicRules"]].copy()
base["DeltaToQuali"] = base["LapTimeSeconds"] - base["QualiRef"]

# Compare each lap with the median of the same driver's NON-yellow laps in the same stint
grp = ["Year", "Round", "Driver", "Stint"]
stint_median = base[~base["IsYellow"]].groupby(grp)["DeltaToQuali"].median().rename("StintMedian")
base = base.join(stint_median, on=grp)
base["DevFromStint"] = base["DeltaToQuali"] - base["StintMedian"]

yellow = base[base["IsYellow"]]
normal = base[~base["IsYellow"]]
print(f"Yellow laps: {len(yellow):,} ({len(yellow) / len(base):.1%} of laps passing the basic rules)")
print("\nYellow laps vs the stint's normal laps (s):")
print(yellow["DevFromStint"].describe().round(2).to_string())
print("\nFor comparison — normal laps vs their own stint median (s):")
print(normal["DevFromStint"].describe().round(2).to_string())
print(f"\nShare more than 1 s slower than the stint median: yellow {(yellow['DevFromStint'] > 1).mean():.1%} "
      f"| normal {(normal['DevFromStint'] > 1).mean():.1%}")

Yellow laps: 1,700 (2.4% of laps passing the basic rules)

Yellow laps vs the stint's normal laps (s):
count    1695.00
mean        0.49
std         1.78
min        -3.38
25%        -0.26
50%         0.17
75%         0.77
max        27.32

For comparison — normal laps vs their own stint median (s):
count    67850.00
mean         0.08
std          0.94
min         -5.42
25%         -0.25
50%          0.00
75%          0.30
max         56.50

Share more than 1 s slower than the stint median: yellow 19.5% | normal 7.0%


In [10]:
# Same 107% idea: a race lap slower than 107% of that race's median clean lap is an incident, not pace
race_median = base.groupby(["Year", "Round"])["LapTimeSeconds"].transform("median")
base["IsSlowOutlier"] = base["LapTimeSeconds"] > 1.07 * race_median

print(f"Slow outliers (>107% of race median): {base['IsSlowOutlier'].sum():,} "
      f"({base['IsSlowOutlier'].mean():.2%})")
print(base.loc[base["IsSlowOutlier"], "DeltaToQuali"].describe().round(2).to_string())

after = base[~base["IsSlowOutlier"]]
print(f"\nLaps left after outliers: {len(after):,}")
print(f"Laps still faster than the driver's qualifying lap: {(after['DeltaToQuali'] < 0).sum()}")
print(after[after["DeltaToQuali"] < 0].groupby(["Year", "EventName", "Driver"]).size()
      .sort_values(ascending=False).head(10).to_string())

print("\nDeltaToQuali after cleaning (s):")
print(after["DeltaToQuali"].describe().round(2).to_string())
print("\nLaps per season:")
print(after.groupby("Year").size().to_string())

Slow outliers (>107% of race median): 80 (0.12%)
count    80.00
mean     21.54
std      14.82
min      10.17
25%      12.24
50%      14.07
75%      22.09
max      62.51

Laps left after outliers: 69,470
Laps still faster than the driver's qualifying lap: 17
Year  EventName         Driver
2023  Qatar Grand Prix  ZHO       17

DeltaToQuali after cleaning (s):
count    69470.00
mean         6.17
std          1.68
min         -1.11
25%          5.03
50%          6.07
75%          7.23
max         14.83

Laps per season:
Year
2022    13326
2023    16619
2024    19745
2025    19780


In [11]:
# --- Retirement, decided from the data (not the status text) ---
race_last = laps.groupby(["Year", "Round"])["LapNumber"].transform("max")
driver_last = laps.groupby(["Year", "Round", "Driver"])["LapNumber"].transform("max")
laps["Retired"] = driver_last < race_last - 3
laps["IsRetirementLap"] = laps["Retired"] & (laps["LapNumber"] >= driver_last - 1)

per_driver = laps.drop_duplicates(["Year", "Round", "Driver"])
print(f"Drivers flagged as retired: {per_driver['Retired'].sum()}")
print("Their statuses:")
print(per_driver.loc[per_driver["Retired"], "Status"].fillna("(blank)").value_counts().head(10).to_string())

# --- Qualifying reference still above 107% of the fastest in that race ---
pole_ref = ref.groupby(["Year", "Round"])["QualiRef"].min().rename("PoleRef")
laps = laps.drop(columns=["PoleRef"], errors="ignore").merge(pole_ref, on=["Year", "Round"], how="left")
laps["BadQualiRef"] = laps["QualiRef"] > 1.07 * laps["PoleRef"]

# --- Target (new column) ---
laps["DeltaToQuali"] = laps["LapTimeSeconds"] - laps["QualiRef"]

rules = {
    "Wet race or wet quali":                    laps["ExcludeRace"].fillna(False),
    "Quali reference still >107%":              laps["BadQualiRef"],
    "Lap 1 (standing start)":                   laps["LapNumber"] <= 1,
    "Pit in-lap or out-lap":                    laps["PitInTime"].notna() | laps["PitOutTime"].notna(),
    "SC / VSC / red-flag lap":                  laps["IsNeutralised"],
    "Restart lap":                              laps["IsRestartLap"],
    "Yellow-flag lap":                          laps["IsYellow"],
    "Not a dry compound":                       ~laps["Compound"].isin(DRY),
    "Bad or missing timing":                    ~laps["IsAccurate"].eq(True) | laps["FastF1Generated"].eq(True)
                                                | laps["LapTimeSeconds"].isna(),
    "Last 2 laps before retirement":            laps["IsRetirementLap"],
}

keep = pd.Series(True, index=laps.index)
rows = []
for name, mask in rules.items():
    removed = int((keep & mask).sum())
    keep &= ~mask
    rows.append((name, removed, int(keep.sum())))

# Slow outliers: judged against the median of laps that survived every rule above
race_median = laps["LapTimeSeconds"].where(keep).groupby([laps["Year"], laps["Round"]]).transform("median")
slow = keep & (laps["LapTimeSeconds"] > 1.07 * race_median)
keep &= ~slow
rows.append(("Slow outlier (>107% of race median)", int(slow.sum()), int(keep.sum())))

laps["IsClean"] = keep
waterfall = pd.DataFrame(rows, columns=["Rule (applied in order)", "Laps removed", "Laps left"])
print(f"\nStarting laps: {len(laps):,}")
print(waterfall.to_string(index=False))
print(f"\n✅ Clean laps: {keep.sum():,} ({keep.mean():.1%} of all laps)")

Drivers flagged as retired: 212
Their statuses:
Status
Retired             134
Collision damage     12
Accident             11
Collision             9
Engine                7
Power Unit            5
                      4
Gearbox               4
Undertray             3
Hydraulics            3

Starting laps: 101,291
            Rule (applied in order)  Laps removed  Laps left
              Wet race or wet quali         19817      81474
        Quali reference still >107%             0      81474
             Lap 1 (standing start)          1450      80024
              Pit in-lap or out-lap          4999      75025
            SC / VSC / red-flag lap          3512      71513
                        Restart lap          1222      70291
                    Yellow-flag lap          1730      68561
                 Not a dry compound           534      68027
              Bad or missing timing            56      67971
      Last 2 laps before retirement            94      67877
Slow outli

In [12]:
laps = laps.sort_values(["Year", "Round", "Driver", "LapNumber"]).reset_index(drop=True)
stint_keys = ["Year", "Round", "Driver", "Stint"]

# Whole-number versions of FastF1's decimal columns
laps["LapNo"] = laps["LapNumber"].astype("Int64")
laps["TyreAge"] = laps["TyreLife"].astype("Int64")
laps["StintNo"] = laps["Stint"].astype("Int64")

# Race distance and fuel proxy
laps["RaceLaps"] = laps.groupby(["Year", "Round"])["LapNumber"].transform("max").astype(int)
laps["LapsRemaining"] = laps["RaceLaps"] - laps["LapNo"]   # more laps left = more fuel on board
laps["RaceProgress"] = laps["LapNo"] / laps["RaceLaps"]

# GreenFlagTyreLaps: tyre age counting only laps raced at full speed.
# = laps the tyre had done before this stint (e.g. in qualifying) + green laps so far in the stint
green = (~laps["IsNeutralised"]).astype(int)
green_so_far = green.groupby([laps[k] for k in stint_keys]).cumsum()
age_before_stint = laps.groupby(stint_keys)["TyreLife"].transform("first") - 1
laps["GreenFlagTyreLaps"] = age_before_stint + green_so_far

# Train / validation / test, by season (never random)
laps["Split"] = laps["Year"].map({2022: "train", 2023: "train", 2024: "val", 2025: "test"})

print(laps.loc[laps["IsClean"], ["LapNo", "TyreAge", "GreenFlagTyreLaps", "LapsRemaining",
                                  "RaceProgress", "TrackTemp", "AirTemp", "DeltaToQuali"]]
      .describe().round(2).T.to_string())

                     count   mean    std   min   25%   50%   75%    max
LapNo              67823.0  31.49  17.73   2.0  16.0  31.0  45.0   78.0
TyreAge            67753.0  15.39   10.1   1.0   8.0  13.0  21.0   78.0
GreenFlagTyreLaps  67753.0  14.61   9.86   1.0   7.0  13.0  20.0   77.0
LapsRemaining      67823.0  29.97  17.87   0.0  15.0  30.0  44.0   74.0
RaceProgress       67823.0   0.51   0.28  0.03  0.27  0.51  0.76    1.0
TrackTemp          67823.0  36.89   8.31  16.7  31.0  36.3  43.8   57.5
AirTemp            67823.0  25.08   4.45  14.2  21.9  25.7  28.8   37.2
DeltaToQuali       67823.0   6.16   1.67 -1.11  5.03  6.06  7.22  14.83


In [13]:
clean = laps[laps["IsClean"]].copy()

# GreenFlagTyreLaps can never exceed TyreAge
too_big = (clean["GreenFlagTyreLaps"] > clean["TyreAge"]).sum()
print(f"Laps where GreenFlagTyreLaps > TyreAge (should be 0): {too_big}")
diff = clean["TyreAge"] - clean["GreenFlagTyreLaps"]
print(f"Laps where the two differ (tyre has done SC/VSC laps): {(diff > 0).sum():,} ({(diff > 0).mean():.1%})")
print(f"Typical difference when they differ: {diff[diff > 0].median():.0f} laps (max {diff.max():.0f})")

# Missing values in the model inputs
feature_cols = ["Compound", "Circuit", "TyreAge", "GreenFlagTyreLaps", "LapNo",
                "LapsRemaining", "RaceProgress", "TrackTemp", "AirTemp", "DeltaToQuali"]
print("\nMissing values per column:")
print(clean[feature_cols].isna().sum().to_string())

# The split
split_summary = clean.groupby("Split").agg(
    Laps=("DeltaToQuali", "size"),
    Races=("Round", lambda r: clean.loc[r.index, ["Year", "Round"]].drop_duplicates().shape[0]),
    MedianDelta=("DeltaToQuali", "median"),
).loc[["train", "val", "test"]]
print("\nSplit summary:")
print(split_summary.round(2).to_string())

# Every circuit in val/test should also appear in train (otherwise the model has never seen it)
train_circuits = set(clean.loc[clean["Split"] == "train", "Circuit"])
for s in ["val", "test"]:
    unseen = sorted(set(clean.loc[clean["Split"] == s, "Circuit"]) - train_circuits)
    print(f"Circuits in {s} but not in train: {unseen if unseen else 'none'}")

Laps where GreenFlagTyreLaps > TyreAge (should be 0): 0
Laps where the two differ (tyre has done SC/VSC laps): 16,932 (25.0%)
Typical difference when they differ: 3 laps (max 14)

Missing values per column:
Compound              0
Circuit               0
TyreAge              70
GreenFlagTyreLaps    70
LapNo                 0
LapsRemaining         0
RaceProgress          0
TrackTemp             0
AirTemp               0
DeltaToQuali          0

Split summary:
        Laps  Races  MedianDelta
Split                           
train  28983     33         5.94
val    19478     20         6.23
test   19362     20         6.08
Circuits in val but not in train: ['Imola', 'Monaco', 'Shanghai']
Circuits in test but not in train: ['Imola', 'Monaco', 'Montréal', 'Shanghai']


In [14]:
model_cols = [
    "Year", "Round", "EventName", "Circuit", "Split", "Driver", "Team", "StintNo", "Compound",
    "LapNo", "TyreAge", "GreenFlagTyreLaps", "LapsRemaining", "RaceLaps", "RaceProgress",
    "TrackTemp", "AirTemp", "QualiRef", "QualiRefSource", "LapTimeSeconds", "DeltaToQuali",
]

# 1) The modelling table: clean laps only
clean_path = PROCESSED_DIR / "laps_clean.parquet"
clean[model_cols].to_parquet(clean_path, index=False)

# 2) Every lap with all flags: the simulator needs pit laps and SC/VSC laps too
all_path = PROCESSED_DIR / "laps_all_flags.parquet"
laps.to_parquet(all_path, index=False)

# 3) One row per race: circuit, distance, split and exclusion reasons
race_info = race_flags.merge(
    laps.drop_duplicates(["Year", "Round"])[["Year", "Round", "RaceLaps", "Split"]],
    on=["Year", "Round"], how="left",
)
race_path = PROCESSED_DIR / "races.parquet"
race_info.to_parquet(race_path, index=False)

# Read back and check nothing was lost
assert pd.read_parquet(clean_path).shape == clean[model_cols].shape
assert pd.read_parquet(all_path).shape[0] == len(laps)
assert pd.read_parquet(race_path).shape[0] == 92
for p in [clean_path, all_path, race_path]:
    print(f"✅ {p.name}: {p.stat().st_size / 1e6:.1f} MB")

✅ laps_clean.parquet: 0.7 MB
✅ laps_all_flags.parquet: 2.8 MB
✅ races.parquet: 0.0 MB


## Notes & findings – 02 Cleaning and features

**Inputs**
- 92 races (101,291 laps) + qualifying lap data for all 92 sessions (`src/download_quali.py`, laps validated by checking sector times sum to the lap time – removed e.g. a 73.670 s glitch "lap" at Jeddah 2022).
- `Circuit` column: 25 circuits; "Miami Gardens" (2025) mapped to "Miami" (now 4 races).

**Qualifying reference (`QualiRef`)**
- Results-table and lap-data bests agree within 0.01 s for 97.1% of 1,789 drivers; the 41 differences are all lap data faster (likely laps deleted for track limits, which need race-control messages we didn't load) → official results used first.
- Order of preference: results (1,764) → lap data (26; fixes 2025 Miami) → team-mate (7). 107% rule: references slower than 107% of the fastest replaced by lap data (1, Sargeant Saudi 2023: 128.510 → 89.721 s) or the team-mate (29). Source recorded in `QualiRefSource`.

**Race-level exclusions (leakage-safe: decided from qualifying, not race results)**
- Wet race (any inter/wet laps): 12. Wet qualifying (any inter/wet qualifying laps; rain alone not counted): 12. Both: 5. **Excluded: 19 of 92.**
- Cross-check: 9 of 11 wet-quali races had median race-vs-quali gaps far below normal (Las Vegas 2025 −14.5 s, Silverstone 2022 −7.4 s, Canada 2022 −6.6 s). São Paulo 2022 (4.66 s) and Silverstone 2024 (5.32 s) looked normal – kept the simple single rule.
- Rain-only qualifying sessions (8, e.g. Azerbaijan 2025, median gap 4.00 s) look normal → rule confirmed.
- Unusual but kept: Qatar 2023 (median gap 2.89 s; 18-lap tyre limit meant flat-out stints). 17 laps (Zhou, Qatar 2023) still faster than qualifying – 0.02%, left in.

**Lap-level rules (waterfall, applied in order): 101,291 → 67,823 clean laps (67.0%)**
- Wet race/quali −19,817 · quali ref >107% −0 · lap 1 −1,450 · pit in/out −4,999 · SC/VSC/red flag −3,512 · restart lap −1,222 · yellow flag −1,730 · not dry compound −534 · bad timing −56 · last 2 laps before retirement −94 · slow outlier (>107% of race median) −54.
- Yellow flags: median effect only +0.17 s vs the stint, but 19.5% of yellow laps were >1 s slow (vs 7.0% of normal laps) → excluded. Not usable as a feature: yellow flags aren't known in advance.
- Retirement decided from data (last lap >3 laps short of the race), not status text: 212 drivers, all genuine retirements.

**Features (all known at prediction time)**
- `TyreAge`, `GreenFlagTyreLaps` (tyre age excluding SC/VSC laps), `LapNo`, `LapsRemaining` (fuel proxy), `RaceProgress`, `TrackTemp`, `AirTemp`, `Compound`, `Circuit`.
- `GreenFlagTyreLaps` differs from `TyreAge` on 25.0% of clean laps (typically by 3 laps, max 14).
- TyreAge reaches 78 (Monaco 2024: lap-1 red flag, most drivers did not stop again).
- 70 laps (0.1%) have no tyre age → dropped in Stage 4.
- Target `DeltaToQuali`: median 6.06 s, IQR 5.03–7.22 s, range −1.11 to 14.83 s.

**Split (by season)**
- Train 2022–23: 28,983 laps, 33 races (median target 5.94 s). Val 2024: 19,478 laps, 20 races (6.23 s). Test 2025: 19,362 laps, 20 races (6.08 s).
- Circuits unseen in training: val – Imola, Monaco, Shanghai; test – Imola, Monaco, Montréal, Shanghai → refit the final model on 2022–24 before testing (leaves only Montréal unseen) and report test error for seen vs unseen circuits.

**Saved (git-ignored, `data/processed/`)**
- `laps_clean.parquet` (0.7 MB, modelling table) · `laps_all_flags.parquet` (2.8 MB, every lap + flags for the simulator) · `races.parquet` (one row per race).